# Multimodal ML Test Notebook

This notebook demonstrates multimodal machine learning for testing the context retrieval persona with mixed data types (text, numerical, categorical).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Generate synthetic multimodal e-commerce product data
np.random.seed(42)
n_samples = 2000

# Product categories and subcategories
categories = ['Electronics', 'Clothing', 'Home & Garden', 'Books', 'Sports']
electronics_subs = ['Smartphones', 'Laptops', 'Headphones', 'Tablets']
clothing_subs = ['Shirts', 'Pants', 'Dresses', 'Shoes']
home_subs = ['Furniture', 'Kitchen', 'Decor', 'Tools']
books_subs = ['Fiction', 'Non-fiction', 'Textbooks', 'Comics']
sports_subs = ['Equipment', 'Apparel', 'Footwear', 'Accessories']

# Generate product data
data = []
for i in range(n_samples):
    category = np.random.choice(categories)
    
    # Subcategory based on category
    if category == 'Electronics':
        subcategory = np.random.choice(electronics_subs)
    elif category == 'Clothing':
        subcategory = np.random.choice(clothing_subs)
    elif category == 'Home & Garden':
        subcategory = np.random.choice(home_subs)
    elif category == 'Books':
        subcategory = np.random.choice(books_subs)
    else:  # Sports
        subcategory = np.random.choice(sports_subs)
    
    # Price based on category (with some noise)
    if category == 'Electronics':
        base_price = np.random.uniform(200, 1500)
    elif category == 'Clothing':
        base_price = np.random.uniform(20, 200)
    elif category == 'Home & Garden':
        base_price = np.random.uniform(30, 500)
    elif category == 'Books':
        base_price = np.random.uniform(10, 50)
    else:  # Sports
        base_price = np.random.uniform(25, 300)
    
    price = round(base_price + np.random.normal(0, base_price * 0.1), 2)
    
    # Rating (influenced by price and category)
    if category == 'Electronics':
        rating_base = 4.2
    elif category == 'Books':
        rating_base = 4.3
    else:
        rating_base = 4.0
    
    rating = round(np.clip(rating_base + np.random.normal(0, 0.3), 1.0, 5.0), 1)
    
    # Number of reviews (correlated with rating and price)
    review_factor = rating / 5.0 * (1 + np.log10(price / 100))
    num_reviews = int(np.random.exponential(50 * review_factor))
    
    # Brand (simplified)
    if category == 'Electronics':
        brand = np.random.choice(['Apple', 'Samsung', 'Sony', 'LG', 'Generic'])
    elif category == 'Clothing':
        brand = np.random.choice(['Nike', 'Adidas', 'H&M', 'Zara', 'Generic'])
    else:
        brand = np.random.choice(['BrandA', 'BrandB', 'BrandC', 'Generic'])
    
    # Generate product title (text feature)
    if category == 'Electronics':
        adjectives = ['Premium', 'High-Quality', 'Advanced', 'Professional', 'Wireless']
        titles = [f'{subcategory}', f'Portable {subcategory}', f'Smart {subcategory}']
    elif category == 'Clothing':
        adjectives = ['Comfortable', 'Stylish', 'Casual', 'Formal', 'Trendy']
        titles = [f'{subcategory}', f'Designer {subcategory}', f'Classic {subcategory}']
    elif category == 'Books':
        adjectives = ['Bestselling', 'Award-winning', 'Popular', 'Educational', 'Inspiring']
        titles = [f'{subcategory} Book', f'{subcategory} Novel', f'{subcategory} Guide']
    else:
        adjectives = ['Professional', 'Durable', 'High-Performance', 'Premium', 'Lightweight']
        titles = [f'{subcategory}', f'Pro {subcategory}', f'Sport {subcategory}']
    
    adj = np.random.choice(adjectives)
    title_base = np.random.choice(titles)
    title = f'{adj} {brand} {title_base}'
    
    # Product description (text feature)
    descriptions = [
        f'High-quality {subcategory.lower()} perfect for daily use. Features advanced technology and durable construction.',
        f'Premium {subcategory.lower()} with excellent performance. Highly rated by customers worldwide.',
        f'Professional grade {subcategory.lower()} designed for optimal results. Trusted by experts.',
        f'Innovative {subcategory.lower()} combining style and functionality. Perfect for modern lifestyle.',
        f'Top-rated {subcategory.lower()} offering exceptional value. Customer favorite with proven results.'
    ]
    description = np.random.choice(descriptions)
    
    # Target: Customer satisfaction (high/low) based on rating and value
    value_score = rating / (price / 100)  # Rating per $100
    satisfaction_prob = 1 / (1 + np.exp(-(value_score - 0.8)))  # Sigmoid
    customer_satisfaction = 'High' if np.random.random() < satisfaction_prob else 'Low'
    
    data.append({
        'product_title': title,
        'product_description': description,
        'category': category,
        'subcategory': subcategory,
        'brand': brand,
        'price': price,
        'rating': rating,
        'num_reviews': num_reviews,
        'customer_satisfaction': customer_satisfaction
    })

# Create DataFrame
multimodal_data = pd.DataFrame(data)
print(f"Multimodal dataset shape: {multimodal_data.shape}")
print(f"\nTarget distribution:")
print(multimodal_data['customer_satisfaction'].value_counts())
multimodal_data.head()

In [ ]:
# Data exploration and visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Price distribution by category
multimodal_data.boxplot(column='price', by='category', ax=axes[0, 0])
axes[0, 0].set_title('Price Distribution by Category')
axes[0, 0].set_xlabel('Category')
axes[0, 0].set_ylabel('Price ($)')

# Rating vs Price scatter
scatter = axes[0, 1].scatter(multimodal_data['price'], multimodal_data['rating'], 
                           c=multimodal_data['customer_satisfaction'].map({'High': 1, 'Low': 0}),
                           alpha=0.6, cmap='RdYlBu')
axes[0, 1].set_xlabel('Price ($)')
axes[0, 1].set_ylabel('Rating')
axes[0, 1].set_title('Price vs Rating (Color: Satisfaction)')
plt.colorbar(scatter, ax=axes[0, 1])

# Number of reviews distribution
axes[0, 2].hist(multimodal_data['num_reviews'], bins=50, alpha=0.7, edgecolor='black')
axes[0, 2].set_xlabel('Number of Reviews')
axes[0, 2].set_ylabel('Frequency')
axes[0, 2].set_title('Distribution of Number of Reviews')
axes[0, 2].set_xlim(0, 500)  # Focus on main distribution

# Customer satisfaction by category
satisfaction_by_category = pd.crosstab(multimodal_data['category'], multimodal_data['customer_satisfaction'])
satisfaction_by_category.plot(kind='bar', ax=axes[1, 0], color=['red', 'green'])
axes[1, 0].set_title('Customer Satisfaction by Category')
axes[1, 0].set_xlabel('Category')
axes[1, 0].set_ylabel('Count')
axes[1, 0].legend(title='Satisfaction')
axes[1, 0].tick_params(axis='x', rotation=45)

# Brand distribution
top_brands = multimodal_data['brand'].value_counts().head(10)
top_brands.plot(kind='bar', ax=axes[1, 1], color='skyblue')
axes[1, 1].set_title('Top 10 Brands by Product Count')
axes[1, 1].set_xlabel('Brand')
axes[1, 1].set_ylabel('Product Count')
axes[1, 1].tick_params(axis='x', rotation=45)

# Rating distribution by satisfaction
high_sat = multimodal_data[multimodal_data['customer_satisfaction'] == 'High']['rating']
low_sat = multimodal_data[multimodal_data['customer_satisfaction'] == 'Low']['rating']

axes[1, 2].hist([high_sat, low_sat], bins=20, alpha=0.7, label=['High Satisfaction', 'Low Satisfaction'],
               color=['green', 'red'], edgecolor='black')
axes[1, 2].set_xlabel('Rating')
axes[1, 2].set_ylabel('Frequency')
axes[1, 2].set_title('Rating Distribution by Customer Satisfaction')
axes[1, 2].legend()

plt.tight_layout()
plt.show()

print("\nDataset Statistics:")
print(multimodal_data.describe())

In [ ]:
# Text analysis - examine product titles and descriptions
print("Sample Product Titles:")
print(multimodal_data['product_title'].head(10).tolist())

print("\nSample Product Descriptions:")
print(multimodal_data['product_description'].head(5).tolist())

# Text length analysis
multimodal_data['title_length'] = multimodal_data['product_title'].str.len()
multimodal_data['description_length'] = multimodal_data['product_description'].str.len()

print(f"\nText Length Statistics:")
print(f"Title length - Mean: {multimodal_data['title_length'].mean():.1f}, Std: {multimodal_data['title_length'].std():.1f}")
print(f"Description length - Mean: {multimodal_data['description_length'].mean():.1f}, Std: {multimodal_data['description_length'].std():.1f}")

# Word frequency analysis
from collections import Counter
import re

def extract_words(text_series):
    all_words = []
    for text in text_series:
        words = re.findall(r'\b\w+\b', text.lower())
        all_words.extend(words)
    return all_words

title_words = extract_words(multimodal_data['product_title'])
description_words = extract_words(multimodal_data['product_description'])

print(f"\nMost common words in titles:")
title_counter = Counter(title_words)
for word, count in title_counter.most_common(10):
    print(f"{word}: {count}")

print(f"\nMost common words in descriptions:")
desc_counter = Counter(description_words)
for word, count in desc_counter.most_common(10):
    print(f"{word}: {count}")

In [ ]:
# Feature correlation analysis
# Create correlation matrix for numerical features
numerical_features = ['price', 'rating', 'num_reviews', 'title_length', 'description_length']
correlation_matrix = multimodal_data[numerical_features].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
           square=True, linewidths=0.5)
plt.title('Correlation Matrix of Numerical Features')
plt.tight_layout()
plt.show()

# Categorical feature analysis
print("\nCategorical Feature Value Counts:")
categorical_features = ['category', 'subcategory', 'brand']
for feature in categorical_features:
    print(f"\n{feature.upper()}:")
    print(multimodal_data[feature].value_counts().head(8))

In [ ]:
# Prepare data for multimodal ML
# Split into train and test sets
train_data, test_data = train_test_split(multimodal_data, test_size=0.2, 
                                        random_state=42, stratify=multimodal_data['customer_satisfaction'])

print(f"Training set size: {len(train_data)}")
print(f"Test set size: {len(test_data)}")
print(f"\nTraining set target distribution:")
print(train_data['customer_satisfaction'].value_counts())
print(f"\nTest set target distribution:")
print(test_data['customer_satisfaction'].value_counts())

# Remove temporary columns
train_data = train_data.drop(['title_length', 'description_length'], axis=1)
test_data = test_data.drop(['title_length', 'description_length'], axis=1)

print(f"\nFinal dataset features:")
print(list(train_data.columns))
print(f"\nData types:")
print(train_data.dtypes)

In [ ]:
# Traditional ML baseline for comparison
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

print("Training baseline Random Forest model...")

# Prepare features for baseline model
baseline_train = train_data.copy()
baseline_test = test_data.copy()

# Encode categorical variables
le_category = LabelEncoder()
le_subcategory = LabelEncoder()
le_brand = LabelEncoder()

baseline_train['category_encoded'] = le_category.fit_transform(baseline_train['category'])
baseline_train['subcategory_encoded'] = le_subcategory.fit_transform(baseline_train['subcategory'])
baseline_train['brand_encoded'] = le_brand.fit_transform(baseline_train['brand'])

baseline_test['category_encoded'] = le_category.transform(baseline_test['category'])
baseline_test['subcategory_encoded'] = le_subcategory.transform(baseline_test['subcategory'])
baseline_test['brand_encoded'] = le_brand.transform(baseline_test['brand'])

# Simple text features (length only for baseline)
baseline_train['title_len'] = baseline_train['product_title'].str.len()
baseline_train['desc_len'] = baseline_train['product_description'].str.len()
baseline_test['title_len'] = baseline_test['product_title'].str.len()
baseline_test['desc_len'] = baseline_test['product_description'].str.len()

# Select features for baseline
baseline_features = ['price', 'rating', 'num_reviews', 'category_encoded', 
                    'subcategory_encoded', 'brand_encoded', 'title_len', 'desc_len']

X_train_baseline = baseline_train[baseline_features]
X_test_baseline = baseline_test[baseline_features]
y_train = baseline_train['customer_satisfaction']
y_test = baseline_test['customer_satisfaction']

# Train baseline model
rf_baseline = RandomForestClassifier(n_estimators=100, random_state=42)
rf_baseline.fit(X_train_baseline, y_train)

# Baseline predictions
y_pred_baseline = rf_baseline.predict(X_test_baseline)
baseline_accuracy = (y_pred_baseline == y_test).mean()

print(f"\nBaseline Random Forest Accuracy: {baseline_accuracy:.4f}")
print("\nBaseline Classification Report:")
print(classification_report(y_test, y_pred_baseline))

# Feature importance
feature_importance = pd.DataFrame({
    'feature': baseline_features,
    'importance': rf_baseline.feature_importances_
}).sort_values('importance', ascending=False)

print("\nFeature Importance (Baseline):")
print(feature_importance)

In [ ]:
# AutoGluon Multimodal comparison would go here
# This cell demonstrates the data format that AutoGluon multimodal expects

print("Data prepared for AutoGluon Multimodal:")
print(f"\nTraining data shape: {train_data.shape}")
print(f"Target column: 'customer_satisfaction'")
print(f"\nText features: product_title, product_description")
print(f"Categorical features: category, subcategory, brand")
print(f"Numerical features: price, rating, num_reviews")

print("\nSample of multimodal data:")
display_cols = ['product_title', 'category', 'brand', 'price', 'rating', 'customer_satisfaction']
print(train_data[display_cols].head())

print("\n" + "="*50)
print("READY FOR AUTOGLUON MULTIMODAL TRAINING")
print("="*50)
print("\nThis dataset contains:")
print("✅ Text data (product_title, product_description)")
print("✅ Categorical data (category, subcategory, brand)")
print("✅ Numerical data (price, rating, num_reviews)")
print("✅ Classification target (customer_satisfaction: High/Low)")
print("\nAutoGluon MultiModalPredictor can automatically handle all these data types!")